In [1]:
import os
from pathlib import Path

os.chdir(Path.cwd().parent)

print(Path.cwd())

d:\private\ai-research-paper-assistant


In [5]:
# ============================================================
# 10. MAIN
# ============================================================
from src.utils.helpers import build_scifact_data, run_scifact_retrieval, compare_retrievers, load_scifact
from src.database.pgvector_storage import Repository
from src.embedder.ollama_embedder import OllamaEmbedder
from src.reranker.cross_encoder_reranker import ReRanker
from src.retriever.bm25 import BM25
from src.retriever.hybrid import HybridSearch
import pandas as pd

if __name__ == "__main__":

    # --------------------------------------------------------
    # Load benchmark
    # --------------------------------------------------------

    corpus_ds, queries_ds, qrels_ds = (
        load_scifact()
    )

    corpus, test_queries, qrels = (
        build_scifact_data(
            corpus_ds,
            queries_ds,
            qrels_ds
        )
    )

    print(
        f"Test queries with GT: "
        f"{len(test_queries)}"
    )


    # --------------------------------------------------------
    # Your project components
    # --------------------------------------------------------

    repository = Repository()

    embedder = OllamaEmbedder(dimensions=2046, num_ctx=8192)

    reranker = ReRanker(
        model_name="BAAI/bge-reranker-v2-m3",
        max_length=512,
        device="cpu"
    )


    # --------------------------------------------------------
    # INGEST ONLY ONCE
    # --------------------------------------------------------

    # Uncomment first time only.
    #
    # ingest_scifact(
    #     corpus=corpus,
    #     repository=repository,
    #     embedder=embedder,
    #     batch_size=64
    # )


    # --------------------------------------------------------
    # BM25
    #
    # Important:
    # BM25 must be fitted against ALL SciFact corpus documents,
    # not Dense candidates.
    # --------------------------------------------------------

    all_documents = (
        repository.get_all_chunks_test(
            benchmark="scifact"
        )
    )

    bm25 = BM25()

    bm25.fit(
        all_documents
    )


    # --------------------------------------------------------
    # Hybrid search
    # --------------------------------------------------------

    hybrid_searcher = HybridSearch(
        repository=repository,
        embedder=embedder,
        bm25=bm25
    )


    # --------------------------------------------------------
    # Run benchmark
    # --------------------------------------------------------

    benchmark_results = (
        run_scifact_retrieval(
            test_queries=test_queries,
            qrels=qrels,

            repository=repository,
            embedder=embedder,

            hybrid_searcher=
                hybrid_searcher,

            reranker=reranker,

            top_k=10,

            candidate_k=100,

            rerank_candidate_k=50,
        )
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    result_df = compare_retrievers(
        benchmark_results,
        top_k=10
    )

    print("\n===== SCIFACT RESULTS =====")
    print(result_df)


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    result_df.to_csv(
        "scifact_retrieval_metrics.csv"
    )

    detail_df = pd.DataFrame(
        benchmark_results["details"]
    )

    detail_df.to_json(
        "scifact_retrieval_details.json",
        orient="records",
        indent=2
    )

    print("\nSaved evaluation results.")

Corpus  : 5183
Queries : 1109
Qrels   : 339
Test queries with GT: 300


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6357.78it/s]


[1/300] Query 1
[2/300] Query 3
[3/300] Query 5
[4/300] Query 13
[5/300] Query 36
[6/300] Query 42
[7/300] Query 48
[8/300] Query 49
[9/300] Query 50
[10/300] Query 51
[11/300] Query 53
[12/300] Query 54
[13/300] Query 56
[14/300] Query 57
[15/300] Query 70
[16/300] Query 72
[17/300] Query 75
[18/300] Query 94
[19/300] Query 99
[20/300] Query 100
[21/300] Query 113
[22/300] Query 115
[23/300] Query 118
[24/300] Query 124
[25/300] Query 127
[26/300] Query 128
[27/300] Query 129
[28/300] Query 130
[29/300] Query 132
[30/300] Query 133
[31/300] Query 137
[32/300] Query 141
[33/300] Query 142
[34/300] Query 143
[35/300] Query 146
[36/300] Query 148
[37/300] Query 163
[38/300] Query 171
[39/300] Query 179
[40/300] Query 180
[41/300] Query 183
[42/300] Query 185
[43/300] Query 198
[44/300] Query 208
[45/300] Query 212
[46/300] Query 213
[47/300] Query 216
[48/300] Query 217
[49/300] Query 218
[50/300] Query 219
[51/300] Query 230
[52/300] Query 232
[53/300] Query 233
[54/300] Query 236
[55/3

KeyboardInterrupt: 